# Single agent approach
- a single agent ias responsible for creating story

In [10]:
import asyncio
import dotenv
from autogen_ext.models.openai import  OpenAIChatCompletionClient
import os


model_client = OpenAIChatCompletionClient(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    model_info={
        "family": "llama",
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "structured_output": True,
    },
)



In [11]:
from autogen_agentchat.agents import AssistantAgent

story_teller_agent = AssistantAgent(
    name = "story_agent",
    model_client=model_client,
    system_message="Your are a creative writter. Genarate a short story about a brrave knight and a dragon."
)


In [ ]:
from autogen_agentchat.messages import TextMessage


async def test_agent():
    task = TextMessage(
        content="write a short story aboout a brave kight and a dragon",
        source="user"
    )

    result = await story_teller_agent.run(task=task)
    print(result.messages[-1].content)

await test_agent()

 

In [13]:
plot_agent =  AssistantAgent(
    name ='plot_agent',
    model_client=model_client,
    system_message="you create engagin plot for the stories. Focus n the knight journey"
)

character_agent =  AssistantAgent(
    name = 'character_agent',
    model_client=model_client,
    system_message='you develop characters. Describe then kight and the dragon in detials, including there motivation and backgound'
)

ending_agent = AssistantAgent(
    name='ending_agent',
    model_client=model_client,
    system_message='you write engaging ending. conclude the story with a twist'
)

# Round Robin Chat in AutoGen

## What is Round Robin Chat?

In AutoGen, a **Round Robin Chat** is a multi-agent conversation pattern where multiple agents take turns speaking in a **fixed order**, similar to people sitting around a round table.

Think of it as a meeting where each participant gets a chance to speak one after another.

---

## Simple Example

Suppose you have three agents:

1. **Research Agent** – Finds information and gathers facts.
2. **Writer Agent** – Creates content based on the research.
3. **Reviewer Agent** – Reviews the content and suggests improvements.

The conversation flow follows a cyclic order:

```text
User
  |
  v
Research Agent
  |
  v
Writer Agent
  |
  v
Reviewer Agent
  |
  v
Research Agent
  |
  v
Writer Agent
  |
  v
Reviewer Agent
  |
  v
... continues until the task is completed
```

This fixed sequence of turns is why it is called **Round Robin**.

---

 max_turns defines the maximum number of times agents are allowed to speak before the conversation automatically stops.

In [14]:

from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat


teams_chat = RoundRobinGroupChat(
    participants=[plot_agent, character_agent, ending_agent],
    max_turns=3
)

async def test_team():
    task = TextMessage( 
        content="write a short story aboout a brave kight and a dragon",
        source='user'
    )

    result = await teams_chat.run(task=task);
    for each_agent_message in result.messages:
        print(f'{each_agent_message.source}: {each_agent_message.content}')

await test_team()


user: write a short story aboout a brave kight and a dragon
plot_agent: **The Quest for the Golden Scepter**

In the land of Eridoria, where the sun dipped into the horizon and painted the sky with hues of crimson and gold, Sir Valoric, a gallant knight, embarked on a perilous journey. His quest was to vanquish a fierce dragon, Tharros, who had been terrorizing the kingdom of Alderan. The once-peaceful realm was now plagued by the beast's fiery wrath, and its people lived in constant fear.

Sir Valoric, with his unwavering sense of justice and unshakeable courage, accepted the challenge. He donned his armor, adorned with the symbol of his noble family, and mounted his trusty steed, Galahad. As he rode towards the dragon's lair, the wind whispered ancient prophecies in his ear, and the earth trembled beneath his feet.

Upon arriving at the foreboding mountains, where Tharros was said to reside, Sir Valoric encountered a wise old wizard, Zephyr. The sage revealed to the knight that the d